## Notebook 06 — prototipo del grafo LangGraph

Objetivo: conectar Ingesta → Analista → Redactor → Revisor (stub por
ahora) en un StateGraph funcional, y comprobar que el pipeline
completo corre de extremo a extremo sobre los 3 documentos reales.

No se redefine ningún agente aquí — todos se importan desde
src_agents/, ya graduados. Este notebook solo construye y prueba el
grafo que los conecta.

In [ ]:
# Celda — setup de rutas (mismo patrón que los notebooks anteriores)
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

### Paso 1: confirmar que los tres agentes reales importan bien

Antes de construir el grafo, verificamos que no hay ningún problema
de import pendiente.

In [ ]:
from src_agents.agents.ingestion import agente_ingesta
from src_agents.agents.analyst import agente_analista
from src_agents.agents.redactor_v1 import agente_redactor

print("Los tres agentes importan correctamente")

### Paso 2: construir el StateGraph, con el Revisor stub

Añadimos el stub del Revisor (aprueba siempre, sin llamar a ningún
modelo) para poder probar el grafo completo hoy sin depender de
haber terminado feature/reviewer-agent.

In [ ]:
# Celda — stub temporal del Revisor
from src_agents.models.state import RevisionResultado, EstadoPipeline

def agente_revisor_stub(estado: EstadoPipeline) -> dict:
    """Stub temporal: aprueba siempre, sin llamar a ningún modelo. Se
    sustituye por agente_revisor real en cuanto esté terminado
    (feature/reviewer-agent), sin tocar el resto del grafo."""
    return {"review": RevisionResultado(valido=True, incidencias=[])}

### Paso 3: definir el grafo — nodos y aristas

Un StateGraph se construye en 3 fases: (1) declaras qué tipo de
estado maneja, (2) registras cada agente como nodo con add_node,
(3) conectas el orden con add_edge. START y END son constantes
especiales de LangGraph que marcan la entrada y salida del grafo.

In [ ]:
from langgraph.graph import StateGraph, START, END

grafo = StateGraph(EstadoPipeline)

grafo.add_node("ingesta", agente_ingesta)
grafo.add_node("analista", agente_analista)
grafo.add_node("redactor", agente_redactor)
grafo.add_node("revisor", agente_revisor_stub)

grafo.add_edge(START, "ingesta")
grafo.add_edge("ingesta", "analista")
grafo.add_edge("analista", "redactor")
grafo.add_edge("redactor", "revisor")
grafo.add_edge("revisor", END)

pipeline = grafo.compile()
print("Grafo compilado correctamente")

### Paso 4: ejecutar el pipeline completo, con manejo de fallo

Si Groq corta a mitad de camino (como ayer), queremos saber en qué
nodo se paró y qué alcanzó a procesar antes, no solo ver un traceback
en rojo y quedarnos sin nada.

In [ ]:
from groq import RateLimitError

estado_inicial = {"uploaded_files": [str(DATA_DIR)]}

try:
    resultado_final = pipeline.invoke(estado_inicial)
    print("Pipeline completo, sin errores")
    print(f"Documentos: {len(resultado_final['documents'])}")
    print(f"Conceptos del Analista: {len(resultado_final['analysis'].datos)}")
    print(f"Borrador generado: {len(resultado_final['draft'])} caracteres")
    print(f"Revisión: {resultado_final['review']}")
except RateLimitError as e:
    print("Se agotó la cuota de Groq a mitad del pipeline.")
    print(f"Detalle: {e}")
    print("\nEsto no es un fallo del grafo — el diseño en sí ya está validado")
    print("hasta donde llegó. Reintentar cuando se resetee la cuota.")